# 🔍 04 — Why does the TGN raise an alert? TempME and feature-level explanations

The final TGN (`tgn_1h_final_v2`, seed 0) flags contagion risk: after coin *i* shocks at hour *t*, will coin *j*
follow within *k* hours? This notebook asks **why**, at two levels:

* **Events (TempME).** Which of the recent shock events the model attends to drive an alert? TempME (Chen & Ying,
  NeurIPS 2023) is ported to our TGN following the official code (`cryptograph.explain.tempme`; route B of
  `docs/model_plan.md` §4): temporal motifs around both coins, an information-bottleneck explainer, and importance
  weights that multiply the TGN's attention. It is compared with random, recency, the TGN's own attention and a mask
  optimised separately for every alert (the best any event-level explanation can do).
* **Features (integrated gradients).** Which inputs of the prediction (each coin's current state, its memory and
  recent events, the shock event itself) drive an alert?

1. Reference rankings. 2. How the explanations are produced. 3. Faithfulness. 4. Why event-level explanations are
weak for this model. 5. Feature-level explanations. 6. Plausibility against network rankings. 7. Case studies:
LUNA (May 2022), FTX (Nov 2022), the 10-Oct-2025 liquidation cascade.

**Protocol.** The explainer is trained on validation-period events; everything reported is on the test period (the
riskiest 1% of alerts at *k* = 6 h, plus a random sample of test events) and on the case windows. Explanations are
applied to the real model: the captured inputs reproduce its predictions exactly (checked below).

In [1]:
import sys, json
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"
V2 = "../data/processed/v2"

In [2]:
EXP = Path("../reports/explain")
summary = json.loads((EXP / "summary.json").read_text())
fid = pd.read_csv(EXP / "fidelity.csv")
feat = pd.read_csv(EXP / "feature_attributions.csv")
plaus = pd.read_csv(EXP / "plausibility.csv")
motif = pd.read_csv(EXP / "motif_categories.csv")
credit = pd.read_csv(EXP / "coin_credit_tempme.csv")
cases = json.loads((EXP / "case_studies.json").read_text())
runs = pd.read_csv("../runs/results.csv")
print("explained model:", summary["target_run"])
display(runs[runs["experiment"] == "tempme_tgn_1h"][["run_id", "commit", "dirty", "status"]].tail(1))

explained model: 20261004-065801_tgn_1h_final_v2_0_8f5d566


,run_id,commit,dirty,status
66,20261006-172137_tempme_tgn_1h_0_1238a1e,1238a1e,False,ok


## 1. Reference rankings

In [3]:
rk = pd.read_csv(f"{V2}/rankings.csv")
meta = json.loads(Path(f"{V2}/rankings.json").read_text())
print("window:", meta["window"], "| note:", meta["note"])
display(rk[["coin", "sector", "dy_net", "dy_pagerank", "shock_pagerank", "eigenvector"]].round(3).head(10))
display(rk[["dy_net", "dy_pagerank", "shock_pagerank", "eigenvector", "mst_degree"]].corr(method="spearman").round(2))

window: ['2024-04-02 08:00:00', '2026-09-27 23:00:00'] | note: evaluation benchmark for explanations (TempME); computed on the test window, never used as a model input


,coin,sector,dy_net,dy_pagerank,shock_pagerank,eigenvector
0,LINK,DeFi / oracle,21.418,0.040,0.037,0.037
1,DOT,smart-contract platform,20.186,0.040,0.036,0.037
2,AVAX,smart-contract platform,18.468,0.040,0.038,0.037
3,ADA,smart-contract platform,16.799,0.039,0.037,0.037
4,ETH,smart-contract platform,16.466,0.039,0.037,0.037
5,DOGE,meme,16.340,0.039,0.035,0.037
6,ETC,smart-contract platform,16.135,0.039,0.039,0.036
7,ARB,layer 2,14.225,0.038,0.032,0.036
8,SHIB,meme,12.544,0.038,0.037,0.036
9,SOL,smart-contract platform,10.694,0.037,0.037,0.036


,dy_net,dy_pagerank,shock_pagerank,eigenvector,mst_degree
dy_net,1.00,1.00,0.74,1.00,0.79
dy_pagerank,1.00,1.00,0.74,1.00,0.79
shock_pagerank,0.74,0.74,1.00,0.72,0.61
eigenvector,1.00,1.00,0.72,1.00,0.77
mst_degree,0.79,0.79,0.61,0.77,1.00


*What this shows:* the rankings explanations will be compared with. The correlation-based ones (DY NET,
spillover PageRank, eigenvector) are the same ranking at hourly resolution; the time-ordered shock-transmission
PageRank is related but different (ρ ≈ 0.73). Agreement with the time-ordered one matters most (notebook 02 §2–3).

## 2. How the explanations are produced

For every explained event the TGN attends to the **10 most recent events** of each of the two coins (up to 20
events). An explanation scores those events. Removing an event = giving it weight 0 in the attention (TempME's
way of applying an explanation to a TGN); the node memory is left as is. The captured inputs must reproduce the
model's predictions exactly, and integrated gradients must add up to the logit difference (completeness).

In [4]:
print(f"rebuilt vs model logits, max abs difference: {summary['rebuild_max_abs_logit_diff']:.1e}")
print(f"integrated gradients, max completeness gap (logit units): {summary['ig_completeness_max_gap']:.3f}")
h = pd.DataFrame(summary["history"])
display(h.round(4))

rebuilt vs model logits, max abs difference: 2.4e-07
integrated gradients, max completeness gap (logit units): 0.009


,epoch,loss,holdout_char_k3
0,0,0.6043,0.1525
1,1,0.5616,0.1502
2,2,0.5240,0.1555
3,3,0.5195,0.1591
4,4,0.5191,0.1586
5,5,0.5192,0.1570
6,6,0.5192,0.1577
7,7,0.5189,0.1557
8,8,0.5186,0.1616
9,9,0.5189,0.1566


*What this shows:* The captured inputs reproduce the TGN's own logits to 2×10⁻⁷, and the integrated-gradient attributions add up
to the logit difference within 0.009, so every number below describes the real model, not an approximation of it.
The TempME explainer was trained on 10,000 validation events (official settings: learning rate 10⁻³, β = 0.5, prior
sparsity 0.3, temperature 0.07). Its loss fell in the first 3 epochs and then flattened; the hold-out characterization
stayed between 0.150 and 0.162 throughout, and training stopped at epoch 13. **The explainer found very little
structure to learn**; section 4 shows why.

## 3. Faithfulness: do the chosen events carry the prediction?

Keep only the top-*s* events (**Fidelity−**, lower = the explanation alone suffices) or remove them (**Fidelity+**,
higher = they were necessary). Both are normalised by the effect of removing *all* recent events. Characterization =
harmonic mean of Fidelity+ and 1 − Fidelity− (Amara et al., 2022).

In [5]:
f = fid.copy()
fig = px.line(f, x="kept_events", y="characterization", color="explainer", facet_col="events", markers=True,
              title="Characterization score by number of events in the explanation (higher is better)")
fig.update_layout(height=420)
fig.show()
k3 = f[f["kept_events"] == 3].pivot_table(index="explainer", columns="events",
                                          values=["fid_plus_norm", "fid_minus_norm", "characterization"])
display(k3.round(3))
print("effect of removing all recent events (mean |dp|):", f.groupby("events")["empty_effect"].first().round(4).to_dict())

characterization                fid_minus_norm  \
events                     random test events top alerts random test events   
explainer                                                                     
TGN attention                           0.143      0.147              0.865   
TempME                                  0.160      0.157              0.848   
optimised mask (per alert)              0.249      0.248              0.755   
random                                  0.151      0.152              0.855   
recency                                 0.153      0.192              0.856   

                                           fid_plus_norm             
events                     top alerts random test events top alerts  
explainer                                                            
TGN attention                   0.859              0.153      0.153  
TempME                          0.846              0.169      0.161  
optimised mask (per alert)      0.754              0.253      0.249  
random                          0.849              0.158      0.153  
recency                         0.811              0.163      0.194

effect of removing all recent events (mean |dp|): {'random test events': 0.0131, 'top alerts': 0.0142}


*What this shows:* **No event-level explanation carries much of this model's prediction.** With 3 of the up to 20 recent events
kept, TempME reaches a characterization of 0.157 on the riskiest test alerts and 0.160 on random test events; random
choice gives 0.152 and 0.151. So TempME is only marginally better than chance. On the top alerts the simple
**recency** rule ("the most recent events matter most") does better (0.192). The **per-alert optimised mask**, which
fits every alert separately and is the best an event-level explanation can do, reaches 0.248. Even that is far from
the 1.0 of a perfect explanation: removing the 3 "best" events changes the prediction by only about a quarter of
what removing all of them does. All curves rise almost linearly with the number of events kept. **For this TGN, the
contribution of recent events is spread almost evenly over them**, and no small set of past events explains an alert.

## 4. Why event-level explanations are weak for this model

Two diagnostics: how selective the TGN's attention is (normalised entropy 1 = perfectly uniform over the recent
events), and whether the motif types around real events differ from those in a time-shuffled stream (TempME's null
model). If attention is uniform and motifs look random, there is little for any event-level explainer to find.

In [6]:
print(f"TGN attention over recent events: normalised entropy {summary['attention_norm_entropy']:.4f}, "
      f"mean largest weight {summary['attention_mean_max']:.3f} (uniform over 10 events = 0.100)")
names = {0: "1 step", 1: "2 steps, back to start", 2: "2 steps, new coin", 3: "3 steps a-b-a-b", 4: "3 steps a-b-a-c",
         5: "3 steps a-b-c-a", 6: "3 steps a-b-c-b", 7: "3 steps, 4 different coins"}
m = motif.copy()
m["motif"] = [names[c % 8] + (" (via other end)" if c >= 8 else "") for c in m["category"]]
m = m[(m["real"] > 0.001) | (m["null (time-shuffled)"] > 0.001)]
fig = px.bar(m.melt(id_vars="motif", value_vars=["real", "null (time-shuffled)"], var_name="stream", value_name="share"),
             x="motif", y="share", color="stream", barmode="group", title="Temporal motif types: real vs time-shuffled")
fig.update_layout(height=420)
fig.show()

TGN attention over recent events: normalised entropy 1.0000, mean largest weight 0.100 (uniform over 10 events = 0.100)


*What this shows:* **The TGN's attention over recent events is perfectly uniform**: normalised entropy 1.000, mean largest weight
0.100 for 10 events. The trained model averages its recent events instead of picking specific ones, so "which event
mattered" has almost no answer. **The motif types around real events match the time-shuffled null model**: 83% are
3-step walks through 4 different coins in the real stream vs 86% in the null; the only visible excess is walks
that return to a coin (a→b→c→b: 5.9% vs 3.3%). This is the same conclusion as notebook 02 §8: crypto shocks spread
market-wide within the same hour, not along chains of specific coin pairs, so there are no distinctive temporal
motifs for TempME to find. The negative result is informative: **contagion risk in this market is carried by the
state of the coins, not by particular earlier events**, which section 5 confirms.

## 5. Feature-level explanations: what drives an alert

Integrated gradients on the prediction head for the 6-hour horizon, baseline = training mean. Mean absolute
attribution per input (share of the total), for the riskiest test alerts and for random test events.

In [7]:
top = feat[feat["events"] == "top alerts"].head(15)
fig = px.bar(top[::-1], x="mean_attribution", y="input", orientation="h",
             title="Riskiest 1% of test alerts: mean attribution to the 6h logit (positive = raises the risk)")
fig.update_layout(height=520)
fig.show()
grp = feat[feat["events"].isin(["top alerts", "random test events"])].groupby(["events", "group"])["share_of_abs"].sum().unstack(0)
display(grp.sort_values("top alerts", ascending=False).round(3))

events,random test events,top alerts
group,,
receiver coin,0.612,0.649
source coin,0.275,0.245
event,0.113,0.106


*What this shows:* **An alert is mainly about the receiving coin's own state.** About 65% of the attribution goes to the
receiver's current features, 25% to the source's, and 10% to the shock event itself. The largest single driver is the
receiver's **volume surge relative to its past week** (`volume_rel_168h`, 25% of the total, raising the risk), then
its **24-hour volatility** (12%) and its **latest return** (9%). The memory and recent-event embeddings contribute only
about 7% together, consistent with the ablation in notebook 03 (memory matters little at short horizons). The split
is almost identical for the riskiest alerts and for random events, and across all three crashes, so the model
applies one consistent rule: **a coin that is already trading heavily and moving a lot is the one most likely to be
hit next when another coin shocks**.

## 6. Plausibility: do explanations point at the coins that network analysis calls central?

For each top alert, the coins on the other side of its 3 most important events get credit (weighted by importance).
Raw credit is mostly exposure (coins that shock often appear in many recent-event lists), so the credit is also
divided by the coin's share of all recent events ("lift over exposure"). Spearman correlation with the rankings of
notebook 02 (30 coins).

In [8]:
display(plaus.pivot_table(index="ranking", columns="explainer", values=["spearman_share", "spearman_lift_over_exposure"]).round(2))
c = credit.merge(rk[["coin", "sector", "shock_pagerank", "dy_net"]], on="coin")
fig = px.scatter(c, x="shock_pagerank", y="credit_share", text="coin", color="sector",
                 title="TempME credit share (top alerts) vs time-ordered shock PageRank")
fig.update_traces(textposition="top center")
fig.update_layout(height=480)
fig.show()

spearman_lift_over_exposure                                    \
explainer                    TGN attention TempME optimised mask (per alert)   
ranking                                                                        
dy_net                                0.28  -0.83                      -0.42   
dy_pagerank                           0.27  -0.83                      -0.42   
eigenvector                           0.27  -0.83                      -0.43   
shock_out_lift                        0.13  -0.65                      -0.35   
shock_pagerank                        0.14  -0.65                      -0.34   

               spearman_share                                    
explainer       TGN attention TempME optimised mask (per alert)  
ranking                                                          
dy_net                   0.37  -0.79                      -0.21  
dy_pagerank              0.36  -0.79                      -0.20  
eigenvector              0.36  -0.78                      -0.21  
shock_out_lift           0.21  -0.65                      -0.16  
shock_pagerank           0.22  -0.64                      -0.14

*What this shows:* **TempME's explanations do not point at the coins that network analysis calls central; if anything, the
opposite** (Spearman about −0.8 with Diebold–Yilmaz NET and PageRank, −0.65 with the time-ordered shock PageRank). It
credits idiosyncratic coins most (TRX, WLD, ZEC, BCH, QNT) and BTC and ETH least. The per-alert optimised mask, which
is more faithful, shows the same direction but much weaker (−0.2 to −0.4); the TGN's attention follows exposure
(+0.3). Because TempME is barely more faithful than random here (section 3), its coin credit should not be read as
"these coins cause contagion". A cautious reading: shocks of central coins happen together with everything else and
add little specific information, while a shock of a less connected coin is more distinctive. Systemic importance in
this market is better measured by the network rankings of notebook 02 than by event-level explanations of the TGN.

## 7. Case studies

The five riskiest events (6-hour horizon) in each window, with the three most important recent events according to
TempME and the five largest feature attributions. `label` = did the receiver actually shock within 6 hours.

In [9]:
for case in dict.fromkeys(c["case"] for c in cases):
    rows = [c for c in cases if c["case"] == case]
    print(f"\n=== {case} ===")
    display(pd.DataFrame([{"time": r["time"], "alert": r["event"], "p(6h)": round(r["p"], 3), "label": r["label"],
                           "top recent events (TempME)": "; ".join(f"{e['time'][5:16]} {e['event']}" for e in r["top_recent_events"]),
                           "top features": "; ".join(f"{x['input']} ({x['mean_attribution']:+.2f})" for x in r["top_features"][:3])}
                          for r in rows]))
fa = feat[feat["events"].str.startswith("case")]
display(fa.groupby(["events", "group"])["share_of_abs"].sum().unstack(0).round(3))


=== LUNA collapse (May 2022, train period) ===


,time,alert,p(6h),label,top recent events (TempME),top features
0,2022-05-12 05:00:00,QNT -> ADA,0.575,1,05-12 03:00 ADA -> TRX; 05-12 03:00 QNT -> DOG...,receiver coin: rv_24h (+0.38); receiver coin: ...
1,2022-05-12 05:00:00,POL -> BNB,0.570,1,05-12 04:00 BNB -> TRX; 05-12 03:00 QNT -> BNB...,receiver coin: volume_rel_168h (+0.34); receiv...
2,2022-05-12 04:00:00,XRP -> LTC,0.570,1,05-12 02:00 LTC -> TRX; 05-12 02:00 LTC -> SOL...,receiver coin: rv_24h (+0.31); receiver coin: ...
3,2022-05-12 04:00:00,LTC -> XRP,0.568,0,05-12 02:00 LTC -> TRX; 05-12 02:00 LTC -> SOL...,receiver coin: rv_24h (+0.39); receiver coin: ...
4,2022-05-12 04:00:00,DOGE -> LTC,0.566,1,05-12 02:00 LTC -> TRX; 05-11 22:00 QNT -> DOG...,receiver coin: rv_24h (+0.30); receiver coin: ...



=== FTX collapse (Nov 2022, validation period) ===


,time,alert,p(6h),label,top recent events (TempME),top features
0,2022-11-09 15:00:00,DOGE -> ETH,0.599,1,11-09 09:00 SHIB -> DOGE; 11-09 09:00 QNT -> D...,receiver coin: rv_24h (+0.38); receiver coin: ...
1,2022-11-08 18:00:00,DOGE -> ETH,0.596,0,11-08 17:00 QNT -> ETH; 11-08 17:00 XRP -> DOG...,receiver coin: volume_rel_168h (+0.35); receiv...
2,2022-11-09 15:00:00,DOGE -> SOL,0.594,1,11-09 09:00 SHIB -> DOGE; 11-09 09:00 QNT -> D...,receiver coin: rv_24h (+0.41); receiver coin: ...
3,2022-11-08 18:00:00,LINK -> BNB,0.589,1,11-08 17:00 QNT -> BNB; 11-08 17:00 DOGE -> BN...,receiver coin: volume_rel_168h (+0.39); receiv...
4,2022-11-08 18:00:00,ETH -> BNB,0.588,1,11-08 17:00 QNT -> BNB; 11-08 17:00 DOGE -> BN...,receiver coin: volume_rel_168h (+0.39); receiv...



=== 10-Oct-2025 liquidation cascade (test period) ===


,time,alert,p(6h),label,top recent events (TempME),top features
0,2025-10-10 21:00:00,WLD -> JUP,0.665,1,10-10 20:00 AVAX -> WLD; 10-10 20:00 WLD -> EN...,receiver coin: volume_rel_168h (+0.45); receiv...
1,2025-10-10 21:00:00,WLD -> UNI,0.663,1,10-10 20:00 AVAX -> WLD; 10-10 20:00 UNI -> QN...,receiver coin: volume_rel_168h (+0.45); receiv...
2,2025-10-10 21:00:00,WLD -> ICP,0.655,1,10-10 20:00 AVAX -> WLD; 10-10 20:00 WLD -> EN...,receiver coin: volume_rel_168h (+0.46); receiv...
3,2025-10-10 21:00:00,WLD -> ETC,0.650,1,10-10 20:00 AVAX -> WLD; 10-10 20:00 ETC -> ZE...,receiver coin: volume_rel_168h (+0.46); receiv...
4,2025-10-10 21:00:00,LTC -> SUI,0.646,1,10-10 20:00 LTC -> QNT; 10-10 20:00 SUI -> ZEC...,receiver coin: volume_rel_168h (+0.45); receiv...


events,case: 10-Oct-2025 liquidation cascade (test period),"case: FTX collapse (Nov 2022, validation period)","case: LUNA collapse (May 2022, train period)"
group,,,
event,0.099,0.110,0.107
receiver coin,0.663,0.628,0.651
source coin,0.238,0.262,0.242


*What this shows:* **In all three crashes the model's riskiest alerts were mostly right, for the same reasons.** LUNA (12 May
2022): 4 of the 5 riskiest alerts (p ≈ 0.57) were followed by a shock of the receiver within 6 hours. FTX (8–9 Nov
2022): 4 of 5 (p ≈ 0.59). The 10-Oct-2025 liquidation cascade: 5 of 5 (p ≈ 0.65, its highest risk levels). In every
case the leading attributions are the receiver's 24-hour volatility and volume surge, and the attribution split
(receiver about 65%, source about 25%, event about 10%) is the same as on ordinary test events. TempME's top events
are those of the previous 1–3 hours involving the same coins (for example AVAX → WLD at 20:00 before the 21:00 WLD
alerts on 10 Oct 2025), which matches the recency pattern of section 3.

## Key Takeaways

- **TempME was ported faithfully and applied to the real model** (captured inputs reproduce the TGN exactly; motif
  sampler, information-bottleneck explainer and attention weighting follow the official code).
- **Event-level explanations are weak for this TGN, and we can say why.** TempME is barely better than random
  (characterization 0.157 vs 0.152 with 3 events); even an explanation optimised per alert reaches only 0.25. The
  TGN's attention over recent events is perfectly uniform, and temporal motifs look the same as in a time-shuffled
  stream: shocks spread market-wide within the hour, not along chains (same as notebook 02).
- **What actually drives an alert is the state of the coins.** About 65% of the prediction comes from the receiving
  coin's current state (volume surge vs its past week, 24h volatility, latest return), 25% from the shocking coin and
  10% from the shock event itself; memory and recent events about 7%.
- **The rule is consistent and it worked in real crashes.** The same pattern holds for LUNA, FTX and the
  10-Oct-2025 cascade, where 13 of the 15 riskiest alerts came true within 6 hours.
- **Systemic coins:** TempME's coin credit runs against network centrality (ρ ≈ −0.8), but with low faithfulness it is
  not evidence about which coins transmit contagion; use the network rankings of notebook 02 for that.

**For the paper:** TempME works as a diagnostic. It shows that the contagion signal is not in specific past
events or motifs but in the current state of the coins, which feature attribution then makes interpretable.